In [1]:
! pip install evaluate
! pip install --upgrade transformers
! pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.2/481.2 kB 31.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.29.0
    Uninstalling huggingface-hub-0.29.0:
      Successfully uninstalled huggingface-hub-0.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.0
    Uninstalling transformers-4.47.0:
      Successfully uninstalled transformers-4.47.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 18.3 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
dataset = load_dataset("sahil2801/CodeAlpaca-20k")

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json:   0%|          | 0.00/8.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 20022
    })
})

In [4]:
test_set = dataset['train'].select(range(20022-int(0.1*20022),20022))

In [ ]:
from huggingface_hub import login
login(token = "XXXXXXXXXXXXXXXXX") # replace with your own token

In [6]:
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch

base_model = "TheMockingJay1013/gemma-3-sft-peft"
# torch_dtype = torch.float16
attn_implementation = "eager"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    attn_implementation=attn_implementation
)

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

In [7]:
tokenizer.pad_token = tokenizer.eos_token

# for dare models only. Comment out for the other models 
# from trl import setup_chat_format
# model, tokenizer = setup_chat_format(model, tokenizer) 

In [8]:
responses = []
gt = [e['output'] for e in test_set]

In [9]:
import torch

model.eval()  # Set model to evaluation mode

with torch.no_grad():  # Disable gradient tracking
    for i in range(len(test_set)):
        if i % 100 == 0:
            print(f"Processing index {i}")

        text = (
            "<bos><|im_start|>system\n" + test_set[i]['instruction'] +
            "<|im_end|>\n<|im_start|>user\n<|im_end|>\n" +
            test_set[i]['input'] + "<|im_start|>assistant\n"
        )

        encoded = tokenizer([text], return_tensors="pt").to("cuda")

        outputs = model.generate(
            **encoded,
            max_new_tokens=100,
            temperature=1.0,
            top_k=64,
        )

        t = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
        t = t.split("assistant\n")[-1]
        responses.append(t)


Processing index 0
Processing index 100
Processing index 200
Processing index 300
Processing index 400
Processing index 500
Processing index 600
Processing index 700
Processing index 800
Processing index 900
Processing index 1000
Processing index 1100
Processing index 1200
Processing index 1300
Processing index 1400
Processing index 1500
Processing index 1600
Processing index 1700
Processing index 1800
Processing index 1900
Processing index 2000


In [10]:
! pip install rouge_score
! pip install --upgrade pip
! pip install nltk==3.8.1


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=24e188e5e787622c9636bd1aaca6e2687269cb94ce33d6fdb9c7ceff68752595
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.1 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.2.4
    Uninstalling nltk-3.2.4:
      Successfully uninstalled nltk-3.2.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
preprocessing 0.1.13 requires nltk==3.2.4, bu

In [11]:
# gt = gt[:len(responses)]

In [12]:
import evaluate



# Load metrics
rouge = evaluate.load("rouge")
# meteor = evaluate.load("meteor")
bleu = evaluate.load("bleu")

# Compute metrics
rouge_result = rouge.compute(predictions=responses, references=gt, rouge_types=["rougeL"])
# meteor_result = meteor.compute(predictions=responses, references=gt)
bleu_result = bleu.compute(predictions=responses, references=[[ref] for ref in gt])



In [13]:
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize

def compute_meteor_scores(predictions, references):
    """
    Computes the METEOR scores between prediction and reference sentence pairs.

    Args:
        predictions (list of str): List of predicted sentences.
        references (list of str): List of reference sentences (ground truth).

    Returns:
        float: Average METEOR score across all pairs.
    """
    assert len(predictions) == len(references), "Predictions and references must have the same length."

    scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens = word_tokenize(pred)
        ref_tokens = word_tokenize(ref)
        score = meteor_score([ref_tokens], pred_tokens)
        scores.append(score)

    average_score = sum(scores) / len(scores)
    return average_score

# # Example usage
# responses = [
#     "The cat is on the mat",
#     "There is a cat on the mat"
# ]

# gtx = [
#     "A cat is sitting on the mat",
#     "A cat is on the mat"
# ]

avg = compute_meteor_scores(responses, gt)


In [14]:
# Print results
print("ROUGE-L:", rouge_result["rougeL"])
print("METEOR score:", avg)
print("BLEU:", bleu_result["bleu"])

ROUGE-L: 0.2709806831361747
METEOR score: 0.35266160702615335
BLEU: 0.1706971745709151
